In [0]:
import pandas as pd
import sqlite3 
from pathlib import Path

In [0]:
DATOS = Path('/Workspace/Users/diegoagameroc@gmail.com/Primerospasos/Project_Python/Data/menus_restaurantes')

In [0]:
Archivos = DATOS.glob('*')
lista_df =[]
for archivo in Archivos:
    if archivo.suffix == '.csv':
        print(archivo)
        df=pd.read_csv(archivo,sep=',')
        lista_df.append(df)
        df['Origen']=archivo.stem
    else:
        print('No es CSV')

In [0]:
df_menus = pd.concat(lista_df, ignore_index= True)
df_restaurantes = pd.read_csv('/Workspace/Users/diegoagameroc@gmail.com/Primerospasos/Project_Python/Data/restaurants.csv')


In [0]:
#Profiling
print(len(lista_df))        
print(df_menus.shape)       
print(df_restaurantes.shape) 

In [0]:
# df_menus
print(df_menus.shape)
print(df_menus.dtypes)
print(df_menus.isnull().sum())

In [0]:
# df_restaurantes
print(df_restaurantes.shape)
print(df_restaurantes.dtypes)
print(df_restaurantes.isnull().sum())

EN  DFRESTAURANTES

In [0]:
#1. Mapeo de price range
print(df_restaurantes['price_range'].value_counts(dropna=False))

In [0]:
mapa_precios = {
    '$': 'Económico',
    '$$': 'Moderadamente caro',
    '$$$': 'Caro',
    '$$$$': 'Muy caro',
}


In [0]:
df_restaurantes['rango_precio'] = df_restaurantes['price_range'].map(mapa_precios)

# Después del mapeo
print(df_restaurantes['precio_legible'].value_counts(dropna=False))

In [0]:
# 2. Estandarizar ubicaciones | Split de dirección

partes = df_restaurantes['full_address'].str.split(',')

df_restaurantes['calle'] = partes.str[0].str.strip()

df_restaurantes['ciudad'] = partes.str[-3].str.strip()

df_restaurantes['estado_zip'] = (
    partes.str[-2].str.strip() + ' ' + partes.str[-1].str.strip()
)

display(print(df_restaurantes[['full_address', 'calle', 'ciudad', 'estado_zip']].head()))

In [0]:
# 3. Filtro de calidad
df_restaurantes = df_restaurantes[
    df_restaurantes['score'].notna() & (df_restaurantes['score'] != 0)
].copy()

df_restaurantes['category'] = df_restaurantes['category'].str.lower()

print(df_restaurantes.shape)
print(df_restaurantes['score'].isna().sum()) 
print(df_restaurantes['category'].head())

EN DF MENUS

In [0]:

filas_antes = len(df_menus)
df_menus['price'] = (
    df_menus['price']
    .str.replace(' USD', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)


In [0]:
# 2. Eliminar precios nulos, 0.0 y negativos
df_menus = df_menus[
    df_menus['price'].notna() & (df_menus['price'] > 0)
].copy()

print(f"Filas eliminadas: {filas_antes - len(df_menus):,}")
print(df_menus['price'].dtype)    
print(df_menus['price'].describe())

Optimizacion y cruce 

In [0]:
df_restaurantes_consolidados = df_restaurantes[df_restaurantes['ratings'] > 100].copy()
print(f"Restaurantes consolidados: {len(df_restaurantes_consolidados):,}")

df_master = pd.merge(
    df_restaurantes_consolidados,
    df_menus,
    left_on='id',               # llave en df_restaurantes
    right_on='restaurant_id',   # llave en df_menus
    how='inner',
    suffixes=('_rest', '_menu') 
)
print(df_master.shape)
print(df_master.columns.tolist())

CARGA A BASE DE DATOS

In [0]:
con = sqlite3.connect("delivery_insights.db")


In [0]:
df_master.to_csv('master.csv', sep=',', index=False)

In [0]:
df_master.to_sql('master_food_data', con, if_exists='replace', index=False)


In [0]:
pd.read_sql('SELECT * FROM master_food_data', con).head()


In [0]:
#Penetración Geográfica: ¿Cuáles son las 5 ciudades (ciudad) con la mayor
#cantidad de restaurantes consolidados (los que quedaron en nuestra base)?
#Gráfico sugerido: Gráfico de barras horizontal ordenado de mayor a menor.


In [0]:
query_ciudades = """
SELECT 
ciudad 
,COUNT(DISTINCT id) AS total_restaurantes
FROM master_food_data
WHERE ciudad IS NOT NULL
GROUP BY ciudad
ORDER BY total_restaurantes DESC
LIMIT 5
"""
pd.read_sql(query_ciudades,con)

In [0]:
#Análisis de Oportunidad de Precio: El CEO cree que todos los restaurantes en
#el mercado son caros. Usando los datos, muestre la distribución de la columna
#rango_de_precios a nivel general (¿Cuántos hay en "Económico", "Caro", etc.?).
#Gráfico sugerido: Gráfico de torta (Pie chart) o Donut chart.


In [0]:
query_precios = """
SELECT
    COALESCE(precio_legible, 'Sin Datos') AS rango_precio,
    COUNT(DISTINCT id) AS total_restaurantes
FROM master_food_data
GROUP BY rango_precio
ORDER BY total_restaurantes DESC
"""

pd.read_sql(query_precios,con)

In [0]:
#Estrategia de Menú por Ciudad: Para las 3 ciudades más populares obtenidas
#en la pregunta 1, calcule el precio promedio de los platos en el menú. ¿En
#qué ciudad es más caro comer en promedio? Gráfico sugerido: Gráfico de barras
#comparativo.


In [0]:
query_menu_ciudad = """
WITH top3_ciudades AS (
    SELECT ciudad
    FROM master_food_data
    WHERE ciudad IS NOT NULL
    GROUP BY ciudad
    ORDER BY COUNT(DISTINCT id) DESC
    LIMIT 3
)
SELECT
    m.ciudad,
    ROUND(AVG(m.price), 2) AS precio_promedio,
    COUNT(*) AS total_platos
FROM master_food_data AS m
INNER JOIN top3_ciudades AS t
    ON m.ciudad = t.ciudad
GROUP BY m.ciudad
ORDER BY precio_promedio DESC
"""

df_precio_ciudad = pd.read_sql(query_menu_ciudad, con)
df_precio_ciudad

In [0]:
##La Pregunta del Millón (Correlación Precio-Calidad): ¿Un restaurante más
#caro garantiza una mejor experiencia para el usuario? Escriba un query que
#calcule el puntaje (score) promedio agrupado por rango_de_precios. Utilice un
#gráfico de líneas o de barras para demostrar visualmente al CEO si la calidad
#sube a medida que aumenta la categoría de precio.


In [0]:
query_calidad = """
WITH restaurantes_unicos AS (
    SELECT DISTINCT id, precio_legible, score
    FROM master_food_data
    WHERE precio_legible IS NOT NULL
)
SELECT
    precio_legible AS rango_precio,
    ROUND(AVG(score), 2) AS score_promedio,
    COUNT(*) AS total_restaurantes
FROM restaurantes_unicos
GROUP BY precio_legible
ORDER BY CASE precio_legible
    WHEN 'Económico' THEN 1
    WHEN 'Moderadamente caro' THEN 2
    WHEN 'Caro' THEN 3
    WHEN 'Muy caro' THEN 4
END
"""

df_calidad = pd.read_sql(query_calidad, con)
df_calidad